In [ ]:
import json
import time
import uuid
import subprocess
from pathlib import Path
from typing import Any, Dict, Optional




In [ ]:
LI6800 = r"licor@[fe80::f01c:15ff:feb8:9639%6]" # Change this to your LI-6800's IPV6 SSH address
REMOTE_DIR = "/home/licor/apps/dynamic"
REMOTE_CMD = f"{REMOTE_DIR}/remote_cmd.json"
REMOTE_TMP = f"{REMOTE_DIR}/remote_cmd.json.tmp"
REMOTE_ACK = f"{REMOTE_DIR}/remote_ack.json"

'\naction\n\nA string that can be used to indicate intent.\n\nIn the current BP, it’s only used for one thing: if action is "stop", "quit", "exit" (case-insensitive), the BP will terminate.\n\nAny other value (e.g. "measure") is effectively ignored and the BP proceeds normally.\n\nco2_r\n\nSets the reference CO₂ setpoint (ppm / µmol mol⁻¹) via:\n\nSETCONTROL("CO2_r", sp_co2, "float")\n\nIf wait_for_co2 is true, the BP will wait until measured CO2_r is close to this value (within co2_tol).\n\nqin\n\nSets the incident light (PPFD) setpoint via:\n\nSETCONTROL("Qin", sp_qin, "float")\n\nUnits are whatever your LI-6800 expects for Qin (typically µmol m⁻² s⁻¹ for PPFD).\n\nflow\n\nSets the chamber flow setpoint via:\n\nSETCONTROL("Flow", sp_flow, "float")\n\nUnits depend on your configuration (commonly µmol s⁻¹).\n\ntair\n\nSets the chamber air temperature setpoint via:\n\nSETCONTROL("Tair", sp_tair, "float")\n\nIn °C.\n\nrh_air\n\nSets the chamber relative humidity setpoint via:\n\nSETCONTR

In [ ]:

def _run(cmd: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, capture_output=True, text=True)

def send_and_wait_ack(
    cmd: Dict[str, Any],
    *,
    local_cmd_path: Path = Path("remote_cmd.json"),
    local_ack_path: Path = Path("remote_ack.json"),
    identity_file: Optional[Path] = None,
    timeout_s: float = 120.0,
    poll_s: float = 0.5,
) -> Dict[str, Any]:
    """
    Sends a RemoteEnvMeasure command and blocks until the matching ack is written.

    Requires:
      - RemoteEnvMeasure.py writes remote_ack.json
      - RemoteEnvMeasure.py includes cmd_id in the ack (recommended)
    """

    # 1) Attach unique ID to command
    cmd_id = cmd.get("cmd_id") or str(uuid.uuid4())
    cmd = dict(cmd)
    cmd["cmd_id"] = cmd_id

    # 2) Write local JSON
    local_cmd_path.write_text(json.dumps(cmd, indent=2), encoding="utf-8")

    scp = ["scp"]
    ssh = ["ssh"]
    if identity_file:
        scp += ["-i", str(identity_file)]
        ssh += ["-i", str(identity_file)]

    # 3) Upload to tmp
    r1 = _run(scp + [str(local_cmd_path), f"{LI6800}:{REMOTE_TMP}"])
    if r1.returncode != 0:
        raise RuntimeError(f"SCP upload failed:\n{r1.stderr}")

    # 4) Atomic rename tmp -> cmd
    r2 = _run(ssh + [LI6800, f"mv {REMOTE_TMP} {REMOTE_CMD}"])
    if r2.returncode != 0:
        raise RuntimeError(f"SSH mv failed:\n{r2.stderr}")

    # 5) Poll until ack has our cmd_id
    t0 = time.time()
    last_err = ""
    while time.time() - t0 < timeout_s:
        # fetch ack (small file; simplest approach)
        r3 = _run(scp + [f"{LI6800}:{REMOTE_ACK}", str(local_ack_path)])
        if r3.returncode == 0:
            try:
                ack = json.loads(local_ack_path.read_text(encoding="utf-8"))
                if ack.get("cmd_id") == cmd_id:
                    return ack
            except Exception as e:
                last_err = f"ACK parse error: {e!r}"
        else:
            last_err = r3.stderr.strip()

        time.sleep(poll_s)

    raise TimeoutError(f"Timed out waiting for ack cmd_id={cmd_id}. Last error: {last_err}")




In [ ]:
ack = send_and_wait_ack({
    "action": "measure",
    "co2_r": 399, # CO2 reference, in ppm
    "tair": 25, # temperature of air, in °C
    "rh_air": 50, # relative humidity of air, in %
    "wait_for_co2": True, # wait until CO2 reading stabilizes before logging
    "co2_tol": 5, # CO2 tolerance for "stable" reading; only used if wait_for_co2=True, in ppm
    "wait_s": 1, # wait this long before loging 
    "qin": 0, # light intensity, in μmol/m²/s
    "log": True # log the measurement
})